# Optimizing Vehicle Routing with Genetic Algorithms

This project solves a **Vehicle Routing Problem (VRP)** using a custom Genetic Algorithm built with the Python `DEAP` library.

**The Goal:** Route a fleet of vehicles to visit a set of locations and return to a central depot with the primary objective of minimizing total distance traveled, subject to secondary constraints of balancing the stop count per vehicle (workload equity) and minimizing the standard deviation in distance traveled across the fleet.

**Key Features:**
* Custom evolutionary loop with Elitism and a Hall of Fame mechanism.
* Parameter tuning to analyze the effects of mutation rates and tournament selection pressure.
* Dynamic visualization of routing and algorithm convergence.

## 1. Core Algorithm & Elitism
The core genetic algorithm utilizes **Ordered Crossover** and **Index Shuffling Mutation**. To prevent the algorithm from "forgetting" good solutions during heavy mutation phases, we implement a **Hall of Fame** and **Elitism** — explicitly injecting the all-time best blueprint back into the breeding pool each generation.

## 2. Parameter Sweeps and Diversity Analysis
To understand how our hyperparameters dictate the algorithm's behavior, we run a two-part experiment:

1. **Parameter Sweeps:** We test individual values for Population Size, Mutation Rate, and Tournament Size to measure their direct impact on the final fitness score.
2. **Diversity Over Time:** We test four unique parameter configurations across hundreds of generations. We monitor both the **Best Distance** and the **Population Diversity** (standard deviation of fitness) to map exactly how selection pressure controls the balance between exploration and exploitation.

In [ ]:
#Required Libraries
!pip install matplotlib deap
import random
import numpy as np
import statistics
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms

## Problem Setup & Configuration

The map is a grid with a depot at the center. Locations are randomly generated; vehicles are assigned stops in a round-robin interleave (ith vehicle gets indices i, i+num_vehicles, i+num_vehicles*2, …).

In [ ]:
# --- Configuration ---
num_locations = 40
num_vehicles  = 4
limit         = 100
depot         = (limit / 2, limit / 2)

# Generate random locations
random.seed(0)
locations = [(random.randrange(limit), random.randrange(limit))
             for _ in range(num_locations)]

In [ ]:
#class creation
creator.create("FitnessMin", base.Fitness, weights=(-100.0,-1.0))
creator.create("Individual", list, fitness=creator.FitnessMin)

In [ ]:
#fun creation
toolbox = base.Toolbox()
toolbox.register("indices", random.sample, range(num_locations), num_locations)
toolbox.register("individual",tools.initIterate,creator.Individual,toolbox.indices)
toolbox.register("population",tools.initRepeat,list,toolbox.individual)

## Fitness Function

In [ ]:
# Fitness Fun
def evalVRP(individual):
    dist = [0 for _ in range(num_vehicles)]
    for i in range(num_vehicles):
        px = depot[0]
        py = depot[1]
        for j in range(i,num_locations,num_vehicles):
            p = locations[individual[j]]
            dist[i]+=(abs(px-p[0])+abs(py-p[1]))
            px = p[0]
            py = p[1]
            if (j+num_vehicles)>=num_locations :
                dist[i]+=(abs(px-depot[0])+abs(py-depot[1]))
    total = sum(dist)
    std_dev = statistics.stdev(dist) if len(dist) > 1 else 0.0

    return (total,std_dev)

toolbox.register("evaluate", evalVRP)

## Genetic Operators

In [ ]:
# Genetic Operators
toolbox.register("mate", tools.cxOrdered) #crossover
toolbox.register("mutate", tools.mutShuffleIndexes,indpb = 0.05) #mutation
toolbox.register("select",tools.selTournament,tournsize=10) # selection

In [ ]:
# Plotting Fun
def plot_routes(individual, title="Routes"):
    routes = [[depot] for _ in range(num_vehicles)]
    for i in range(num_locations) :
        j = i%num_vehicles
        p = locations[individual[i]]
        routes[j].append(p)
    for r in routes:
        r.append(depot)

    plt.figure(figsize=(5,4))

    plt.scatter(depot[0],depot[1],color='red',marker='*',s=300,zorder=5,label='depot')

    for i,r in enumerate(routes):
        xs,ys = zip(*r)

        plt.plot(xs,ys,marker='o',linewidth=2,markersize=6,label=f'Vehicle {i+1}')

        for k in range(len(r)-1):
            dx = xs[k+1]-xs[k]
            dy = ys[k+1]-ys[k]
            plt.arrow(xs[k],ys[k],dx,dy,color='black',head_width=0.2,length_includes_head=True,alpha = 0.5,zorder=4)

    plt.title(title)
    plt.xlabel("X Coordinate")
    plt.ylabel("Y Coordinate")
    plt.legend()
    plt.grid(True,linestyle='--',alpha = 0.5)
    '''plt.savefig(f"results/{title.replace(' ', '_').replace('|','').replace('=','')}.png",
                bbox_inches='tight')'''
    plt.show()

## Running the GA

`run_ga()` runs the full evolutionary loop. Set `snapshot_gens` to control which generations get a route plot. The Hall of Fame tracks the single best individual ever seen; elitism re-inserts it at the end of each generation so good solutions are never lost.

In [ ]:
def run_ga(generations=500, pop_size=300, mutpb=0.4, tournsize=5, snapshot_gens=[2, 10, 50, 100, 250, 499]):
    random.seed(42)

    if hasattr(toolbox, 'select'):
        toolbox.unregister('select')
    toolbox.register("select", tools.selTournament, tournsize=tournsize)

    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(1)

    fits = toolbox.map(toolbox.evaluate, pop)
    for fit, ind in zip(fits, pop):
        ind.fitness.values = fit
    hof.update(pop)

    # Data over generations
    history_best_dist = []
    history_diversity = []

    for gen in range(generations):
        offspring = algorithms.varAnd(pop, toolbox, cxpb=0.5, mutpb=mutpb)

        fits = toolbox.map(toolbox.evaluate, offspring)
        for fit, ind in zip(fits, offspring):
            ind.fitness.values = fit

        hof.update(offspring)
        pop = toolbox.select(offspring, k=len(pop))
        pop[-1] = toolbox.clone(hof[0]) # Elitism

        history_best_dist.append(hof[0].fitness.values[0])

        pop_dists = [ind.fitness.values[0] for ind in pop]
        diversity = np.std(pop_dists)
        history_diversity.append(diversity)

        if gen in snapshot_gens:
            plot_routes(hof[0], title=f"Routes at Generation {gen} | Distance = {hof[0].fitness.values[0]}")

    return history_best_dist, history_diversity, hof[0]

history_best_dist, history_diversity, best_ind = run_ga()

## Parameter Experiments

Two-phase study:
- **Phase 1** — independent sweeps of population size, mutation rate, and tournament size (150 generations each).
- **Phase 2** — four full configurations tracked over 300 generations, comparing convergence speed and population diversity.

In [ ]:
def run_experiments():
    # ==========================================
    # PHASE 1: INDIVIDUAL PARAMETER SWEEPS
    # ==========================================
    print("--- Running Phase 1: Parameter Sweeps ---")
    sweep_gens = 150

    # 1. Sweep Population Size (Constant: mutpb=0.4, tournsize=5)
    pop_sizes = [5, 50, 500]
    pop_results = []
    print("Testing Population Sizes...")
    for p in pop_sizes:
        best_hist, _, _ = run_ga(generations=sweep_gens, pop_size=p, mutpb=0.4, tournsize=5, snapshot_gens=[])
        pop_results.append(best_hist[-1]) # Save the final distance

    # 2. Sweep Mutation Rate (Constant: pop_size=300, tournsize=5)
    mut_rates = [0.008, 0.08, 0.8]
    mut_results = []
    print("Testing Mutation Rates...")
    for m in mut_rates:
        best_hist, _, _ = run_ga(generations=sweep_gens, pop_size=300, mutpb=m, tournsize=5, snapshot_gens=[])
        mut_results.append(best_hist[-1])

    # 3. Sweep Tournament Size (Constant: pop_size=300, mutpb=0.4)
    tourn_sizes = [2, 20, 200]
    tourn_results = []
    print("Testing Tournament Sizes...")
    for t in tourn_sizes:
        best_hist, _, _ = run_ga(generations=sweep_gens, pop_size=300, mutpb=0.4, tournsize=t, snapshot_gens=[])
        tourn_results.append(best_hist[-1])

    # Plot Phase 1 Results
    fig1, (ax_pop, ax_mut, ax_tourn) = plt.subplots(1, 3, figsize=(15, 4))

    ax_pop.plot(pop_sizes, pop_results, marker='o', color='blue', linewidth=2)
    ax_pop.set_title("Effect of Population Size")
    ax_pop.set_xlabel("Population Size")
    ax_pop.set_ylabel("Final Best Distance")
    ax_pop.grid(True, linestyle='--', alpha=0.6)

    ax_mut.plot(mut_rates, mut_results, marker='s', color='green', linewidth=2)
    ax_mut.set_title("Effect of Mutation Rate")
    ax_mut.set_xlabel("Mutation Probability (mutpb)")
    ax_mut.grid(True, linestyle='--', alpha=0.6)

    ax_tourn.plot(tourn_sizes, tourn_results, marker='^', color='red', linewidth=2)
    ax_tourn.set_title("Effect of Tournament Size")
    ax_tourn.set_xlabel("Tournament Size")
    ax_tourn.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    # plt.savefig("results/parameter_sweeps.png", bbox_inches='tight')
    plt.show()

    # ==========================================
    # PHASE 2: FITNESS AND DIVERSITY OVER TIME
    # ==========================================
    print("\n--- Running Phase 2: Configuration Deep Dive ---")
    configs = [
        {"name": "Base: Mut=0.4, Tourn=200", "mutpb": 0.4, "tournsize": 200},
        {"name": "Low Selection: Mut=0.4, Tourn=3", "mutpb": 0.4, "tournsize": 3},
        {"name": "Low Mutation: Mut=0.05, Tourn=200", "mutpb": 0.05, "tournsize": 200},
        {"name": "Balanced: Mut=0.2, Tourn=5", "mutpb": 0.2, "tournsize": 5}
    ]

    results = {}
    deep_dive_gens = 300

    for config in configs:
        print(f"Running {config['name']}...")
        best_dist_hist, div_hist, best_ind = run_ga(
            generations=deep_dive_gens,
            pop_size=300,
            mutpb=config['mutpb'],
            tournsize=config['tournsize'],
            snapshot_gens=[]
        )
        results[config['name']] = {
            "fitness": best_dist_hist,
            "diversity": div_hist
        }

    # Plot Phase 2 Results 
    fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    for name, data in results.items():
        ax1.plot(data["fitness"], label=name)
        ax2.plot(data["diversity"], label=name)

    ax1.set_title("Task 2: Fitness Score Analysis (Total Distance)")
    ax1.set_ylabel("Best Distance (Lower is Better)")
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend()

    ax2.set_title("Task 3: Population Diversity Analysis (Std Dev of Fitness)")
    ax2.set_ylabel("Diversity (Spread of distances)")
    ax2.set_xlabel("Generation")
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend()

    plt.tight_layout()
    # plt.savefig("results/fitness_diversity.png", bbox_inches='tight')
    plt.show()

run_experiments();